# 04 — Avaliação e Interpretabilidade

**MedAI Triage** · Fase 6 do desafio

**Este notebook abre o conjunto de teste.** Ele foi mantido lacrado desde a Fase 3: nenhuma decisão de modelo, hiperparâmetro ou limiar foi tomada olhando para ele. É a única estimativa honesta de desempenho que o projeto tem — e ela só pode ser produzida uma vez.

| Seção | Pergunta |
|---|---|
| 1 | Quanto o modelo acerta, e que tipo de erro comete? |
| 2 | Por que recall e não accuracy? |
| 3 | Quão confiável é esse número? |
| 4 | Quais features o modelo usa? |
| 5 | O que o SHAP revela — e o que ele esconde? |
| 6 | Como o modelo explica casos individuais? |
| 7 | Onde ele erra, e por quê? |
| 8 | Isso serve na prática? |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from medai import viz
from medai.config import carregar_config
from medai.data import explore
from medai.models import explain, persist, plots, robustez
from medai.models.evaluate import classificar_com_faixa, metricas_classificacao, resumo_faixas
from medai.models.train import preparar_dados

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
viz.configurar_estilo(interativo=True)

config = carregar_config(RAIZ / "configs" / "breast_cancer.yaml")
pipeline, meta = persist.carregar(config)
dados = preparar_dados(config)

colunas = meta.features_esperadas
X_teste, y_teste = dados.x_teste[colunas], dados.y_teste
X_fundo, _ = dados.treino_mais_validacao()
X_fundo = X_fundo[colunas]

print(f"Modelo:  {meta.algoritmo} ({meta.versao_modelo})")
print(f"Treinado em {meta.n_amostras_treino} amostras · limiar {meta.limiar_decisao}")
print(f"Teste:   {len(X_teste)} amostras, {int(y_teste.sum())} malignas ({y_teste.mean():.1%})")

---
## 1. Desempenho no conjunto de teste

In [ ]:
proba = pipeline.predict_proba(X_teste)[:, 1]
predito = (proba >= meta.limiar_decisao).astype(int)

m_operacional = metricas_classificacao(y_teste, predito, proba)
m_padrao = metricas_classificacao(y_teste, (proba >= 0.5).astype(int), proba)

comparacao = pd.DataFrame({
    f"limiar operacional ({meta.limiar_decisao})": m_operacional,
    "limiar padrão (0,5)": m_padrao,
})
comparacao.loc[["recall", "precision", "f1", "especificidade", "accuracy",
                "roc_auc", "average_precision", "brier",
                "falsos_negativos", "falsos_positivos"]].round(4)

In [ ]:
fig = plots.plot_matriz_confusao(
    y_teste.to_numpy(), predito,
    titulo=f"Conjunto de teste — limiar {meta.limiar_decisao:.3f}",
)

### Leitura — incluindo o que não saiu como planejado

| Métrica | Valor |
|---|---:|
| Recall | **0,9762** |
| Precisão | 0,8723 |
| F1 | 0,9213 |
| Especificidade | 0,9167 |
| Accuracy | 0,9386 |
| ROC-AUC | 0,9970 |
| PR-AUC | 0,9956 |
| Brier | 0,0193 |

**O alvo de recall ≥ 98% não foi atingido: ficou em 97,62%.**

Isso precisa ser dito sem rodeios. O limiar de 0,158 foi calibrado por predições fora de fold sobre 455 amostras para atingir 98% de recall, e no teste entregou 97,62%. Em números absolutos: **1 falso negativo em 42 casos malignos**.

Dois pontos sobre essa diferença:

1. **Com 42 malignos no teste, um único caso vale 2,4 pontos percentuais de recall.** A granularidade da medida não permite distinguir 97,6% de 98,0% — são o mesmo resultado dentro da resolução disponível. Alcançar exatamente 100% exigiria zero falsos negativos, e a diferença entre 1 e 0 aqui é um caso.

2. **Ainda assim, não se deve declarar o alvo atingido.** A estimativa fora de fold era otimista em relação ao desempenho real, o que é informação útil: em produção, a sensibilidade esperada é *próxima de*, não *garantidamente acima de*, 98%.

**O Brier melhorou** em relação à validação (0,0193 contra 0,0306), reforçando a decisão de não recalibrar (ADR-015).

In [ ]:
fig = plots.plot_curvas_desempenho(
    y_teste.to_numpy(), proba,
    limiar=meta.limiar_decisao, taxa_positivos=float(y_teste.mean()),
)

### Por que a curva PR importa mais que a ROC

ROC-AUC de 0,9970 parece quase perfeito — e é enganoso em problemas desbalanceados. A ROC usa a taxa de falsos positivos, que tem os 72 benignos no denominador: mesmo 6 falsos positivos geram uma taxa de apenas 8,3%, e a curva mal se afasta do canto.

A curva PR usa precisão, cujo denominador são os casos **sinalizados**. Os mesmos 6 falsos positivos, entre 47 sinalizados, derrubam a precisão para 0,872. É a métrica que reflete o que o clínico experimenta: de cada 8 casos que o sistema marca como suspeitos, aproximadamente 1 é benigno.

PR-AUC de 0,9956 continua excelente — mas a diferença de enquadramento é o que torna a segunda honesta.

---
## 2. A escolha da métrica, revisitada com dados

In [ ]:
baseline = pd.Series(0, index=y_teste.index)
m_baseline = metricas_classificacao(y_teste, baseline)

argumento = pd.DataFrame({
    "classificador trivial (sempre benigno)": {
        "accuracy": m_baseline["accuracy"], "recall": m_baseline["recall"],
        "cânceres não sinalizados": m_baseline["falsos_negativos"],
    },
    "MedAI Triage": {
        "accuracy": m_operacional["accuracy"], "recall": m_operacional["recall"],
        "cânceres não sinalizados": m_operacional["falsos_negativos"],
    },
}).T
argumento.round(4)

### O argumento fechado

O classificador trivial atinge **63,2% de accuracy** no teste — e deixa passar **todos os 42 cânceres**. O MedAI Triage atinge 93,9% de accuracy e deixa passar 1.

A diferença de accuracy entre os dois é de 31 pontos. A diferença em cânceres não sinalizados é de **42 para 1**.

Nenhuma métrica que trate esses dois sistemas como "diferentes em 31 pontos" está medindo o que importa. Accuracy responde "qual fração das predições está certa"; a pergunta clínica é "quantos cânceres eu deixo passar, e a que custo". Recall responde a primeira metade; precisão e especificidade respondem a segunda.

É por isso que accuracy aparece no relatório mas **nunca orientou nenhuma decisão** deste projeto.

---
## 3. Robustez — quão confiável é o número?

Todo o resultado acima vem de **uma** divisão dos dados. Trocar a semente produz outro conjunto de teste, outro modelo e outro limiar. A pergunta honesta é quanto do desempenho é o modelo e quanto é a sorte da divisão.

In [ ]:
hiper = {k: (None if v == "None" else (float(v) if v.replace(".","",1).isdigit() else v))
         for k, v in meta.hiperparametros.items()}
tabela_robustez = robustez.avaliar_sob_sementes(config, meta.algoritmo, hiper, colunas)
tabela_robustez[["limiar", "recall", "precision", "f1", "roc_auc",
                 "falsos_negativos", "falsos_positivos"]].round(4)

In [ ]:
robustez.resumir(tabela_robustez)

In [ ]:
fig = plots.plot_robustez(tabela_robustez)

### Leitura

| Métrica | Média | Desvio | Amplitude |
|---|---:|---:|---:|
| Recall | 0,9810 | 0,0199 | 0,9524 – 1,0000 |
| Precisão | 0,8707 | 0,0356 | 0,8367 – 0,9302 |
| F1 | 0,9220 | 0,0142 | 0,9011 – 0,9412 |
| ROC-AUC | 0,9970 | 0,0014 | 0,9957 – 0,9990 |

**O recall varia de 95,2% a 100% conforme a divisão** — amplitude de 4,8 pontos, equivalente a dois casos. A média de 98,1% está acima do alvo; a semente que usamos entregou 97,6%.

Isso é a resposta honesta à pergunta "qual o recall do modelo": **aproximadamente 98%, com incerteza de ±2 pontos**. Qualquer número com mais precisão que isso é falsa exatidão.

Note que **ROC-AUC é notavelmente estável** (desvio de 0,0014) enquanto recall e precisão oscilam. Faz sentido: a AUC resume a ordenação sobre todos os limiares, enquanto recall e precisão dependem de um corte específico aplicado a poucos casos de fronteira. É mais uma razão para desconfiar de comparações de modelos baseadas em diferenças pequenas de métricas de limiar.

---
## 4. Quais features o modelo usa

In [ ]:
coeficientes = explain.coeficientes(pipeline)
coeficientes.head(12).round(3)

In [ ]:
fig = plots.plot_coeficientes(coeficientes)

### Um problema sério com os coeficientes

Comparando o sinal de cada coeficiente com a associação univariada da feature (o d de Cohen da Fase 2):

In [ ]:
df_fundo = X_fundo.copy()
df_fundo["diagnosis"] = dados.treino_mais_validacao()[1].map({1: "M", 0: "B"})
d_cohen = explore.separabilidade_cohen_d(df_fundo, config, colunas)

conflito = pd.DataFrame({"coeficiente": coeficientes["coeficiente"], "cohen_d": d_cohen})
conflito["discordam"] = (conflito["coeficiente"] * conflito["cohen_d"]) < 0
discordantes = conflito[conflito["discordam"]].sort_values("cohen_d", ascending=False)
print(f"{len(discordantes)} de {len(conflito)} features com sinal invertido:\n")
discordantes.round(3)

### Leitura — os coeficientes não são interpretáveis aqui

**Nove das trinta features têm coeficiente com sinal oposto à sua associação univariada.**

O caso mais gritante é `compactness_mean`: d de Cohen de **+1,47** (claramente maior em tumores malignos) e coeficiente de **−0,279** (o modelo a usa para *reduzir* a suspeição).

Isso não é erro de implementação — é **supressão por multicolinearidade**, consequência direta do VIF de 3.904 medido na Fase 3. Quando features carregam informação quase idêntica, a regressão distribui pesos entre elas de forma que a soma funcione, e os sinais individuais perdem significado causal. Um coeficiente negativo ali significa "depois de considerar as outras features, esta serve de ajuste", não "compacidade protege contra câncer".

**Consequência prática, e ela é séria:** não se pode mostrar esses coeficientes a um clínico como se fossem efeitos clínicos. Fazê-lo produziria afirmações falsas sobre biologia. A previsão feita na Fase 3 — de que a multicolinearidade destruiria a interpretabilidade dos coeficientes sem prejudicar a acurácia — **se confirmou exatamente**.

É por isso que a UI mostra SHAP, e não coeficientes.

In [ ]:
permutacao = explain.importancia_por_permutacao(
    pipeline, X_teste, y_teste, scoring="recall", random_state=42
)
permutacao.head(12).round(4)

In [ ]:
fig = plots.plot_permutation_importance(permutacao)

### Permutation importance — e a mesma armadilha

A permutation importance mede o que o modelo **usa**, não o que ele declara: embaralha cada feature e mede quanto o recall piora.

Mas ela sofre do mesmo problema sob redundância: embaralhar `radius_mean` quase não degrada a métrica, porque `perimeter_mean` carrega a mesma informação e permanece intacta. **Importância baixa aqui significa "redundante", não "irrelevante"** — e confundir os dois levaria a remover features que, em conjunto, são decisivas.

---
## 5. SHAP

In [ ]:
explicacao = explain.explicar(pipeline, X_teste, x_fundo=X_fundo)
shap_global = explicacao.importancia_global
shap_global.head(12).round(4).to_frame()

In [ ]:
fig = plots.plot_shap_beeswarm(explicacao)

### O ranking individual contraria a expectativa da EDA

A Fase 2 registrou uma expectativa falsificável: `concave points_worst` tinha o maior d de Cohen (2,60) e deveria liderar a importância.

**Ela não lidera.** O topo do SHAP é `texture_worst`, e `concave points_worst` nem aparece entre as seis primeiras.

Antes de concluir que a EDA estava errada, vale testar a explicação alternativa: a diluição por redundância. `concave points_worst` é altamente correlacionada com `concave points_mean`, `concavity_worst` e `compactness_worst` — o crédito se reparte entre elas. `texture_worst`, por outro lado, tem poucas companheiras correlacionadas e fica com seu crédito inteiro.

Se essa explicação estiver certa, **somar o SHAP por grandeza medida deve reordenar o ranking**.

In [ ]:
shap_agrupado = explain.agrupar_por_medida(shap_global)
shap_agrupado.round(4).to_frame()

In [ ]:
fig = plots.plot_shap_agrupado(shap_agrupado)

### Leitura — a expectativa da EDA se confirma, depois de corrigir a diluição

| Grupo | SHAP somado |
|---|---:|
| **Tamanho** (radius/perimeter/area) | **3,85** |
| **Irregularidade** (concavity/compactness/concave points) | **3,13** |
| Textura | 1,57 |
| Simetria | 0,85 |
| Suavidade | 0,61 |
| Dimensão fractal | 0,49 |

`texture_worst` lidera individualmente com 0,99, mas o grupo textura soma apenas 1,57 — **menos da metade do grupo tamanho**, que soma 3,85 distribuído entre nove features correlacionadas.

A leitura individual do SHAP estava **sistematicamente subestimando** os dois sinais mais importantes, exatamente como a Fase 3 previu. A EDA não errou; a medida individual é que é inadequada sob multicolinearidade severa.

**Isso tem consequência direta de produto.** Uma UI que exibisse apenas o SHAP por feature mostraria "textura foi o fator decisivo" quando o fator decisivo é tamanho nuclear. O agrupamento por grandeza não é refinamento estatístico — é o que torna a explicação clinicamente correta.

E a leitura agrupada é coerente com a prática citopatológica: tamanho e irregularidade de contorno são os critérios centrais de malignidade.

In [ ]:
fig = plots.plot_dependencia(explicacao, shap_global.index[0])

---
## 6. Explicações de casos individuais

É o que a UI apresenta ao clínico. Escolhemos quatro casos de propósito: um acerto confiante de cada tipo e os dois erros.

In [ ]:
verdadeiro = y_teste.to_numpy()
grupos = {
    "verdadeiro positivo": np.where((verdadeiro == 1) & (predito == 1))[0],
    "verdadeiro negativo": np.where((verdadeiro == 0) & (predito == 0))[0],
    "falso positivo": np.where((verdadeiro == 0) & (predito == 1))[0],
    "falso negativo": np.where((verdadeiro == 1) & (predito == 0))[0],
}
escolhidos = {
    rotulo: int(idx[np.argmax(np.abs(proba[idx] - meta.limiar_decisao))])
    for rotulo, idx in grupos.items() if len(idx)
}
pd.DataFrame({
    "caso": list(escolhidos),
    "probabilidade": [round(float(proba[i]), 4) for i in escolhidos.values()],
    "n_no_grupo": [len(grupos[r]) for r in escolhidos],
}).set_index("caso")

In [ ]:
for rotulo, indice in escolhidos.items():
    fig = plots.plot_shap_caso(
        explicacao, indice,
        titulo=f"{rotulo.capitalize()} — probabilidade {proba[indice]:.3f}",
    )
    import matplotlib.pyplot as plt
    plt.show()

---
## 7. Análise de erros

In [ ]:
errados = predito != verdadeiro
erros = pd.DataFrame({
    "real": verdadeiro[errados],
    "probabilidade": proba[errados].round(4),
    "distancia_do_limiar": (proba[errados] - meta.limiar_decisao).round(4),
    "tipo": ["falso negativo" if r == 1 else "falso positivo" for r in verdadeiro[errados]],
}, index=X_teste.index[errados]).sort_values("probabilidade")
erros

In [ ]:
# O falso negativo: que valores ele tinha nas features de maior peso?
indice_fn = escolhidos["falso negativo"]
top_features = shap_global.head(6).index.tolist()

perfil = pd.DataFrame({
    "caso_FN": X_teste.iloc[indice_fn][top_features],
    "média_malignos": X_fundo[top_features][dados.treino_mais_validacao()[1] == 1].mean(),
    "média_benignos": X_fundo[top_features][dados.treino_mais_validacao()[1] == 0].mean(),
})
perfil["mais_parecido_com"] = np.where(
    (perfil["caso_FN"] - perfil["média_malignos"]).abs()
    < (perfil["caso_FN"] - perfil["média_benignos"]).abs(),
    "maligno", "benigno",
)
perfil.round(3)

### Leitura dos erros

**Sete erros: 1 falso negativo e 6 falsos positivos.** A proporção reflete a escolha de projeto — o limiar baixo troca falsos negativos por falsos positivos deliberadamente.

**Os falsos positivos são casos de fronteira.** Cinco dos seis têm probabilidade entre 0,19 e 0,34, logo acima do limiar de 0,158. São o custo direto de operar com limiar conservador: casos benignos com alguma característica atípica sobem na fila, um profissional revisa e descarta. Erro recuperável no mesmo dia.

**O falso negativo é o caso que importa.** Probabilidade 0,1166 — apenas 0,041 abaixo do limiar. Diferente do caso visto na Fase 5 (probabilidade 0,0226), **este é um caso de fronteira**: o modelo estava perto de sinalizá-lo.

A explicação SHAP mostra o porquê: `texture_worst` = 20,9 contra média de 25,7 nos malignos, e `texture_mean` = 15,8 contra 19,3. **É um tumor maligno com textura nuclear atipicamente baixa** — e textura é justamente a feature de maior peso individual no modelo.

Isso não é falha de ajuste. É o limite do que 30 medidas morfológicas conseguem capturar: existem tumores malignos cuja aparência citológica se aproxima do benigno. Nenhum limiar os recupera sem inundar a fila de falsos positivos, e nenhuma quantidade de dados de treino elimina essa sobreposição — ela está na biologia, não no modelo.

In [ ]:
faixas = classificar_com_faixa(proba, meta.limiar_decisao, config)
cruzamento = pd.crosstab(faixas, verdadeiro, rownames=["faixa de decisão"], colnames=["maligno"])
cruzamento["total"] = cruzamento.sum(axis=1)
cruzamento

### As faixas de decisão no teste

| Faixa | Benignos | Malignos | Total |
|---|---:|---:|---:|
| Benigno (despriorizado) | 66 | **1** | 67 |
| Zona cinzenta (revisão prioritária) | 4 | **3** | 7 |
| Maligno (priorizado) | 2 | 38 | 40 |

A zona cinzenta está fazendo trabalho real: **captura 3 dos 42 casos malignos** que, sem ela, teriam sido rotulados automaticamente. Esses casos chegam ao revisor marcados como incertos, em vez de receberem um rótulo que o modelo não tem base para dar.

**Mas 1 caso maligno cai na faixa despriorizada.** Em um sistema real, esse caso desceria na fila. Ele seria revisto — todo exame é revisto —, apenas mais tarde.

Esse é o limite honesto do sistema, e precisa estar no material de adoção: **o MedAI Triage reduz, mas não elimina, o risco de atraso diagnóstico.** Ele não é uma rede de segurança; é um reordenador de prioridades que erra em aproximadamente 1 de cada 42 casos malignos.

---
## 8. Síntese

### O que foi medido

| | Valor |
|---|---|
| Recall (teste) | 0,9762 — **abaixo do alvo de 0,98** |
| Recall (5 sementes) | 0,9810 ± 0,0199, amplitude 0,952–1,000 |
| Precisão | 0,8723 |
| ROC-AUC / PR-AUC | 0,9970 / 0,9956 |
| Brier | 0,0193 (melhor que na validação) |
| Erros | 1 falso negativo, 6 falsos positivos |
| Classificador trivial | accuracy 63,2%, **42 cânceres não sinalizados** |

### O que se confirmou

- **A multicolinearidade destruiu a interpretabilidade dos coeficientes**, exatamente como previsto na Fase 3: 9 de 30 features têm sinal invertido em relação à associação univariada.
- **O SHAP individual dilui crédito entre features redundantes.** Agrupado por grandeza, o ranking se reordena e passa a concordar com a EDA e com a prática clínica: tamanho (3,85) e irregularidade (3,13) dominam.
- **A EDA previu o comportamento do modelo** — relações lineares, separabilidade alta, região de sobreposição.

### O que não saiu como planejado

- **O alvo de recall não foi atingido** (97,62% contra 98%). A estimativa fora de fold era otimista. Em produção, esperar "em torno de 98%", não "garantidamente acima".
- Um caso maligno caiu na faixa despriorizada.

### O que permanece desconhecido

- Desempenho em outra instituição, outro equipamento, outra população.
- Viés por subgrupo demográfico — **o dataset não traz essas variáveis**, o que torna a auditoria impossível.
- Comportamento sob deriva temporal.